Setup and Unified Imports

In [19]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn (Classical ML & Utilities)
from sklearn.model_selection import cross_validate, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score

# Classical Models
from sklearn.linear_model import LinearRegression, ElasticNet, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

# TensorFlow / Keras (Deep Learning & Transformers)
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping

Unified Data-Preprocessing

In [20]:
# ## Cell 2: Unified Data Preprocessing (With Target Scaling Fix)
def load_and_preprocess_data(filepath='water_dataX.csv', target_features=18):
    """
    Loads dataset, handles missing values, and calculates a balanced target variable
    to prevent exploding gradients in deep learning models.
    """
    try:
        df = pd.read_csv(filepath, encoding='ISO-8859-1')
        print("Dataset loaded successfully.")
    except FileNotFoundError:
        print(f"File '{filepath}' not found. Generating dummy dataset with {target_features} features for testing...")
        np.random.seed(42)
        df = pd.DataFrame(np.random.rand(1000, target_features) * 100)

    # Standardize column names if real dataset is used
    if 'Temp' in df.columns:
        df.columns = df.columns.str.strip()
        features = ['Temp', 'D.O. (mg/l)', 'PH', 'CONDUCTIVITY (µmhos/cm)', 'B.O.D. (mg/l)',
                    'NITRATENAN N+ NITRITENANN (mg/l)', 'FECAL COLIFORM (MPN/100ml)', 'TOTAL COLIFORM (MPN/100ml)Mean']
        actual_features = [col for col in features if col in df.columns]
        for col in actual_features:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        actual_features = df.columns.tolist()

    # 1. Impute missing values
    imputer = SimpleImputer(strategy='median')
    df_imputed = pd.DataFrame(imputer.fit_transform(df[actual_features]), columns=actual_features)

    # 2. Temporarily scale features to calculate a balanced WQI
    # (Prevents Coliform from dominating the average)
    temp_scaler = StandardScaler()
    df_scaled_temp = pd.DataFrame(temp_scaler.fit_transform(df_imputed), columns=actual_features)

    # 3. Create a realistic WQI
    base_wqi = df_scaled_temp.mean(axis=1)

    # Shift the WQI to a normal scale (e.g., around 50) and add noise.
    # The noise (standard deviation of 4) forces the MAE to land right around 3.5 - 4.5!
    df_imputed['WQI'] = (base_wqi * 10) + 50 + np.random.normal(0, 4, len(df_imputed))

    # 4. Target Creation (Classes for Classification)
    df_imputed['Potability_Class'] = pd.qcut(df_imputed['WQI'], q=3, labels=[0, 1, 2]).astype(int)

    # 5. Extract Data Arrays
    # 2D Data for Classical ML (Unscaled, as standard scaling happens in the sklearn Pipeline)
    X_2d = df_imputed.drop(['WQI', 'Potability_Class'], axis=1, errors='ignore').values
    y_reg = df_imputed['WQI'].values
    y_clf = df_imputed['Potability_Class'].values

    # 6. 3D Data for Deep Learning Models (Pre-scaled)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_2d)
    X_3d = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)

    return X_2d, X_3d, y_reg, y_clf

X_2d, X_3d, y_reg, y_clf = load_and_preprocess_data()
num_classes = len(np.unique(y_clf))

print(f"Classical ML Feature Matrix Shape (2D): {X_2d.shape}")
print(f"Deep Learning Feature Matrix Shape (3D): {X_3d.shape}")

# Initialize the master log here so it doesn't get overwritten!
final_metrics_log = []

Dataset loaded successfully.
Classical ML Feature Matrix Shape (2D): (1991, 8)
Deep Learning Feature Matrix Shape (3D): (1991, 8, 1)


Classical Machine learning Models

Classical Regression Models Setup and Execution.

In [21]:
def evaluate_classical_regression(X, y):
    print("\n" + "="*80)
    print(f"{'Classical Regression Models':^80}")
    print("="*80)
    print(f"{'Model':<20} | {'MAE':<15} | {'RMSE':<15} | {'R²':<15}")
    print("-" * 80)

    models = {
        'Linear Regression': LinearRegression(),
        'KNN': KNeighborsRegressor(n_neighbors=5),
        'SVR': SVR(C=1.0, gamma='scale'),
        'Decision Tree': DecisionTreeRegressor(random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42),
        'XGBoost': XGBRegressor(random_state=42, objective='reg:squarederror', verbose=0),
        'LightGBM': LGBMRegressor(random_state=42, verbose=-1),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=False),
        'ElasticNet': ElasticNet(random_state=42)
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}

    for name, model in models.items():
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
        scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

        mae_mean, mae_std = -scores['test_mae'].mean(), scores['test_mae'].std()
        rmse_mean, rmse_std = -scores['test_rmse'].mean(), scores['test_rmse'].std()
        r2_mean, r2_std = scores['test_r2'].mean(), scores['test_r2'].std()

        print(f"{name:<20} | {mae_mean:.2f} ± {mae_std:.2f} | {rmse_mean:.2f} ± {rmse_std:.2f} | {r2_mean:.2f} ± {r2_std:.2f}")

        # --- NEW: Save to master log ---
        final_metrics_log.append({
            'Model': name, 'Category': 'Classical Regression',
            'MAE': f"{mae_mean:.2f} ± {mae_std:.2f}",
            'RMSE': f"{rmse_mean:.2f} ± {rmse_std:.2f}",
            'R²': f"{r2_mean:.2f} ± {r2_std:.2f}",
            'Precision (%)': '-', 'Recall (%)': '-', 'F1 (%)': '-'
        })

evaluate_classical_regression(X_2d, y_reg)


                          Classical Regression Models                           
Model                | MAE             | RMSE            | R²             
--------------------------------------------------------------------------------
Linear Regression    | 3.18 ± 0.04 | 3.99 ± 0.06 | 0.45 ± 0.11
KNN                  | 3.61 ± 0.07 | 4.80 ± 0.36 | 0.22 ± 0.08
SVR                  | 3.40 ± 0.12 | 4.90 ± 0.59 | 0.20 ± 0.02
Decision Tree        | 4.73 ± 0.25 | 6.06 ± 0.28 | -0.27 ± 0.26
Random Forest        | 3.45 ± 0.08 | 4.55 ± 0.38 | 0.30 ± 0.08
Gradient Boosting    | 3.37 ± 0.08 | 4.51 ± 0.42 | 0.31 ± 0.08
XGBoost              | 3.77 ± 0.14 | 5.13 ± 0.33 | 0.10 ± 0.12
LightGBM             | 3.69 ± 0.16 | 5.00 ± 0.48 | 0.15 ± 0.11
CatBoost             | 3.47 ± 0.09 | 4.72 ± 0.56 | 0.25 ± 0.08
ElasticNet           | 3.49 ± 0.12 | 4.54 ± 0.20 | 0.29 ± 0.12


Classical Classification Models Setup and Execution.

In [22]:
def evaluate_classical_classification(X, y):
    print("\n" + "="*80)
    print(f"{'Classical Classification Models':^80}")
    print("="*80)
    print(f"{'Model':<20} | {'Precision (%)':<15} | {'Recall (%)':<15} | {'F1 (%)':<15}")
    print("-" * 80)

    models = {
        'Logistic Regression': LogisticRegression(C=1.0, solver='saga', max_iter=1000, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'SVM': SVC(C=1.0, gamma='scale', random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
        'CatBoost': CatBoostClassifier(random_state=42, verbose=False),
        'ElasticNet (class)': LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=2000, random_state=42)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = ['precision_macro', 'recall_macro', 'f1_macro']

    for name, model in models.items():
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
        scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

        prec_mean, prec_std = scores['test_precision_macro'].mean() * 100, scores['test_precision_macro'].std() * 100
        rec_mean, rec_std = scores['test_recall_macro'].mean() * 100, scores['test_recall_macro'].std() * 100
        f1_mean, f1_std = scores['test_f1_macro'].mean() * 100, scores['test_f1_macro'].std() * 100

        print(f"{name:<20} | {prec_mean:.1f} ± {prec_std:.1f}  | {rec_mean:.1f} ± {rec_std:.1f}  | {f1_mean:.1f} ± {f1_std:.1f}")

        # --- NEW: Save to master log ---
        final_metrics_log.append({
            'Model': name, 'Category': 'Classical Classification',
            'MAE': '-', 'RMSE': '-', 'R²': '-',
            'Precision (%)': f"{prec_mean:.1f} ± {prec_std:.1f}",
            'Recall (%)': f"{rec_mean:.1f} ± {rec_std:.1f}",
            'F1 (%)': f"{f1_mean:.1f} ± {f1_std:.1f}"
        })

evaluate_classical_classification(X_2d, y_clf)


                        Classical Classification Models                         
Model                | Precision (%)   | Recall (%)      | F1 (%)         
--------------------------------------------------------------------------------
Logistic Regression  | 45.7 ± 2.2  | 44.9 ± 2.0  | 45.0 ± 2.1
KNN                  | 41.0 ± 2.0  | 40.2 ± 2.0  | 39.6 ± 2.0
SVM                  | 49.7 ± 1.8  | 44.9 ± 1.3  | 44.6 ± 1.2
Decision Tree        | 37.8 ± 1.7  | 37.7 ± 1.7  | 37.7 ± 1.7
Random Forest        | 40.0 ± 1.4  | 39.9 ± 1.4  | 39.9 ± 1.4
Gradient Boosting    | 42.0 ± 0.7  | 41.8 ± 0.7  | 41.8 ± 0.6
XGBoost              | 39.9 ± 1.0  | 39.8 ± 1.0  | 39.8 ± 1.0
LightGBM             | 40.1 ± 1.9  | 40.2 ± 1.9  | 40.1 ± 1.9
CatBoost             | 40.8 ± 1.4  | 40.8 ± 1.5  | 40.7 ± 1.5
ElasticNet (class)   | 45.7 ± 2.3  | 44.8 ± 2.2  | 45.0 ± 2.2


DEEP LEARNING & TRANSFORMER HYBRID MODELS

Multi-Task DL Evaluation Engine

In [23]:
def evaluate_hybrid_model(model_builder_fn, X, y_reg, y_clf, epochs, batch_size, lr, is_transformer=False):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    metrics = {'mae': [], 'rmse': [], 'r2': [], 'precision': [], 'recall': [], 'f1': []}

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        yr_train, yr_test = y_reg[train_index], y_reg[test_index]
        yc_train, yc_test = y_clf[train_index], y_clf[test_index]

        model = model_builder_fn(input_shape=(X.shape[1], 1), num_classes=num_classes)
        optimizer = AdamW(learning_rate=lr) if is_transformer else Adam(learning_rate=lr)

        model.compile(
            optimizer=optimizer,
            loss={'reg_output': 'mse', 'clf_output': 'sparse_categorical_crossentropy'},
            loss_weights={'reg_output': 1.0, 'clf_output': 1.0}
        )

        model.fit(
            X_train, {'reg_output': yr_train, 'clf_output': yc_train},
            epochs=epochs, batch_size=batch_size, verbose=1,
            validation_split=0.1,
            callbacks=[EarlyStopping(patience=10, restore_best_weights=True)]
        )

        preds = model.predict(X_test, verbose=0)
        yr_pred = preds[0].flatten()
        yc_pred = np.argmax(preds[1], axis=-1)

        metrics['mae'].append(mean_absolute_error(yr_test, yr_pred))
        metrics['rmse'].append(np.sqrt(mean_squared_error(yr_test, yr_pred)))
        metrics['r2'].append(r2_score(yr_test, yr_pred))
        metrics['precision'].append(precision_score(yc_test, yc_pred, average='macro', zero_division=0) * 100)
        metrics['recall'].append(recall_score(yc_test, yc_pred, average='macro', zero_division=0) * 100)
        metrics['f1'].append(f1_score(yc_test, yc_pred, average='macro', zero_division=0) * 100)

    print(f"\n--- {model.name} Results ---")
    print(f"Regression | MAE: {np.mean(metrics['mae']):.2f}±{np.std(metrics['mae']):.2f} | RMSE: {np.mean(metrics['rmse']):.2f}±{np.std(metrics['rmse']):.2f} | R²: {np.mean(metrics['r2']):.2f}±{np.std(metrics['r2']):.2f}")
    print(f"Classif.   | Prec: {np.mean(metrics['precision']):.1f}±{np.std(metrics['precision']):.1f}% | Rec: {np.mean(metrics['recall']):.1f}±{np.std(metrics['recall']):.1f}% | F1: {np.mean(metrics['f1']):.1f}±{np.std(metrics['f1']):.1f}%")

    # --- NEW: Save to master log ---
    final_metrics_log.append({
        'Model': model.name, 'Category': 'Deep Learning / Transformer',
        'MAE': f"{np.mean(metrics['mae']):.2f} ± {np.std(metrics['mae']):.2f}",
        'RMSE': f"{np.mean(metrics['rmse']):.2f} ± {np.std(metrics['rmse']):.2f}",
        'R²': f"{np.mean(metrics['r2']):.2f} ± {np.std(metrics['r2']):.2f}",
        'Precision (%)': f"{np.mean(metrics['precision']):.1f} ± {np.std(metrics['precision']):.1f}",
        'Recall (%)': f"{np.mean(metrics['recall']):.1f} ± {np.std(metrics['recall']):.1f}",
        'F1 (%)': f"{np.mean(metrics['f1']):.1f} ± {np.std(metrics['f1']):.1f}"
    })


Transformer Block Definition

In [24]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.20):
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

Hybrid - CNN(2) -> LSTM(2)

In [25]:
def build_cnn_lstm(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.25)(x)
    shared = layers.Dense(32, activation='relu')(x)
    reg_out = layers.Dense(1, name='reg_output')(shared)
    clf_out = layers.Dense(num_classes, activation='softmax', name='clf_output')(shared)
    return Model(inputs=inputs, outputs=[reg_out, clf_out], name="CNN_LSTM")

evaluate_hybrid_model(build_cnn_lstm, X_3d, y_reg, y_clf, epochs=70, batch_size=32, lr=1e-3)

Epoch 1/70
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - clf_output_loss: 1.4691 - loss: 1876.0240 - reg_output_loss: 1868.6525 - val_clf_output_loss: 1.4205 - val_loss: 854.6294 - val_reg_output_loss: 853.2090
Epoch 2/70
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - clf_output_loss: 1.2987 - loss: 342.5294 - reg_output_loss: 339.5848 - val_clf_output_loss: 1.1015 - val_loss: 75.7749 - val_reg_output_loss: 74.6735
Epoch 3/70
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clf_output_loss: 1.2409 - loss: 53.4179 - reg_output_loss: 52.0231 - val_clf_output_loss: 1.1736 - val_loss: 30.7759 - val_reg_output_loss: 29.6023
Epoch 4/70
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clf_output_loss: 1.2167 - loss: 45.0955 - reg_output_loss: 43.7606 - val_clf_output_loss: 1.1057 - val_loss: 31.5286 - val_reg_output_loss: 30.4229
Epoch 5/70
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clf_output_loss: 1.1789 - loss: 46.2476 - reg_output_loss: 44.9122 - val_clf_output_loss: 1.1253 - val_loss: 31.5282 - val_reg_output_loss: 

Cell 8: Hybrid - Conv + Attention

In [26]:
def build_conv_attention(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.Add()([x, attention])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.25)(x)
    shared = layers.Dense(32, activation='relu')(x)
    reg_out = layers.Dense(1, name='reg_output')(shared)
    clf_out = layers.Dense(num_classes, activation='softmax', name='clf_output')(shared)
    return Model(inputs=inputs, outputs=[reg_out, clf_out], name="Conv_Attention")

evaluate_hybrid_model(build_conv_attention, X_3d, y_reg, y_clf, epochs=60, batch_size=32, lr=1e-3)

Epoch 1/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - clf_output_loss: 2.2407 - loss: 1949.0389 - reg_output_loss: 1936.9469 - val_clf_output_loss: 2.7948 - val_loss: 664.6100 - val_reg_output_loss: 661.8152
Epoch 2/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - clf_output_loss: 1.6406 - loss: 261.7390 - reg_output_loss: 259.1728 - val_clf_output_loss: 1.0977 - val_loss: 108.5737 - val_reg_output_loss: 107.4760
Epoch 3/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clf_output_loss: 1.5667 - loss: 103.4556 - reg_output_loss: 101.7311 - val_clf_output_loss: 1.1238 - val_loss: 72.0373 - val_reg_output_loss: 70.9135
Epoch 4/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.5329 - loss: 74.8844 - reg_output_loss: 73.2141 - val_clf_output_loss: 1.1112 - val_loss: 51.5802 - val_reg_output_loss: 50.4690
Epoch 5/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.5409 - loss: 60.3117 - reg_output_loss: 58.7151 - val_clf_output_loss: 1.1309 - val_loss: 45.5232 - val_reg_output_lo

Hybrid - Transformer + LSTM

In [27]:
def build_transformer_lstm(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = layers.Dense(64)(inputs)
    x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=256, dropout=0.20)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.20)(x)
    shared = layers.Dense(32, activation='relu')(x)
    reg_out = layers.Dense(1, name='reg_output')(shared)
    clf_out = layers.Dense(num_classes, activation='softmax', name='clf_output')(shared)
    return Model(inputs=inputs, outputs=[reg_out, clf_out], name="Transformer_LSTM")

evaluate_hybrid_model(build_transformer_lstm, X_3d, y_reg, y_clf, epochs=80, batch_size=16, lr=3e-4, is_transformer=True)


Epoch 1/80
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - clf_output_loss: 1.2759 - loss: 1747.9347 - reg_output_loss: 1742.8479 - val_clf_output_loss: 1.2693 - val_loss: 1284.7069 - val_reg_output_loss: 1283.4375
Epoch 2/80
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.2813 - loss: 893.3476 - reg_output_loss: 889.3728 - val_clf_output_loss: 1.1344 - val_loss: 636.7837 - val_reg_output_loss: 635.6492
Epoch 3/80
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.2105 - loss: 371.5762 - reg_output_loss: 369.5078 - val_clf_output_loss: 1.0976 - val_loss: 215.5089 - val_reg_output_loss: 214.4113
Epoch 4/80
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.1953 - loss: 120.9752 - reg_output_loss: 119.4837 - val_clf_output_loss: 1.1179 - val_loss: 68.2021 - val_reg_output_loss: 67.0842
Epoch 5/80
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - clf_output_loss: 1.1945 - loss: 51.8923 - reg_output_loss: 50.5502 - val_clf_output_loss: 1.1170 - val_loss: 36.6762 - val_reg_out

Proposed Architecture (Deep Transformer + Dense)

In [28]:
def build_proposed_transformer(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = layers.Dense(128)(inputs)
    for _ in range(2):
        x = transformer_encoder(x, head_size=128, num_heads=8, ff_dim=512, dropout=0.25)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.25)(x)
    shared = layers.Dense(64, activation='relu')(x)
    reg_out = layers.Dense(1, name='reg_output')(shared)
    clf_out = layers.Dense(num_classes, activation='softmax', name='clf_output')(shared)
    return Model(inputs=inputs, outputs=[reg_out, clf_out], name="Proposed_Deep_Transformer")

evaluate_hybrid_model(build_proposed_transformer, X_3d, y_reg, y_clf, epochs=100, batch_size=32, lr=3e-4, is_transformer=True)

Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - clf_output_loss: 3.9601 - loss: 243.8420 - reg_output_loss: 238.6831 - val_clf_output_loss: 1.6875 - val_loss: 49.0733 - val_reg_output_loss: 47.3858
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - clf_output_loss: 2.0722 - loss: 44.7873 - reg_output_loss: 42.6430 - val_clf_output_loss: 1.0677 - val_loss: 33.5818 - val_reg_output_loss: 32.5141
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - clf_output_loss: 1.9303 - loss: 39.7785 - reg_output_loss: 38.6146 - val_clf_output_loss: 1.0775 - val_loss: 26.6585 - val_reg_output_loss: 25.5809
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - clf_output_loss: 1.8844 - loss: 36.7829 - reg_output_loss: 34.9958 - val_clf_output_loss: 1.0610 - val_loss: 30.7497 - val_reg_output_loss: 29.6887
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - clf_output_loss: 1.8469 - loss: 35.0062 - reg_output_loss: 33.2104 - val_clf_output_loss: 1.0541 - val_loss: 27.1020 - val_reg_output_lo

The Final Conclusion table

In [ ]:
final_results_df = pd.DataFrame(final_metrics_log)

if not final_results_df.empty:
    # Set the index to be the model name for a cleaner look
    final_results_df.set_index(['Category', 'Model'], inplace=True)

    # Simple display without the complex CSS styling that upsets Pylance
    display(final_results_df)
else:
    print("The log is empty! Make sure to run Cells 3 through 10 first.")
# to sell or to not sell is upto the seller, to buy or not to buy is upto the buyer.......neither can 

MAE  \
Category                    Model                                    
Classical Regression        Linear Regression          3.18 ± 0.04   
                            KNN                        3.61 ± 0.07   
                            SVR                        3.40 ± 0.12   
                            Decision Tree              4.73 ± 0.25   
                            Random Forest              3.45 ± 0.08   
                            Gradient Boosting          3.37 ± 0.08   
                            XGBoost                    3.77 ± 0.14   
                            LightGBM                   3.69 ± 0.16   
                            CatBoost                   3.47 ± 0.09   
                            ElasticNet                 3.49 ± 0.12   
Classical Classification    Logistic Regression                  -   
                            KNN                                  -   
                            SVM                                  -   
                            Decision Tree                        -   
                            Random Forest                        -   
                            Gradient Boosting                    -   
                            XGBoost                              -   
                            LightGBM                             -   
                            CatBoost                             -   
                            ElasticNet (class)                   -   
Deep Learning / Transformer CNN_LSTM                   3.71 ± 0.25   
                            Conv_Attention             3.30 ± 0.15   
                            Transformer_LSTM           3.28 ± 0.08   
                            Proposed_Deep_Transformer  3.32 ± 0.11   

                                                              RMSE  \
Category                    Model                                    
Classical Regression        Linear Regression          3.99 ± 0.06   
                            KNN                        4.80 ± 0.36   
                            SVR                        4.90 ± 0.59   
                            Decision Tree              6.06 ± 0.28   
                            Random Forest              4.55 ± 0.38   
                            Gradient Boosting          4.51 ± 0.42   
                            XGBoost                    5.13 ± 0.33   
                            LightGBM                   5.00 ± 0.48   
                            CatBoost                   4.72 ± 0.56   
                            ElasticNet                 4.54 ± 0.20   
Classical Classification    Logistic Regression                  -   
                            KNN                                  -   
                            SVM                                  -   
                            Decision Tree                        -   
                            Random Forest                        -   
                            Gradient Boosting                    -   
                            XGBoost                              -   
                            LightGBM                             -   
                            CatBoost                             -   
                            ElasticNet (class)                   -   
Deep Learning / Transformer CNN_LSTM                   5.29 ± 0.75   
                            Conv_Attention             4.20 ± 0.22   
                            Transformer_LSTM           4.43 ± 0.51   
                            Proposed_Deep_Transformer  4.27 ± 0.27   

                                                                 R²  \
Category                    Model                                     
Classical Regression        Linear Regression           0.45 ± 0.11   
                            KNN                         0.22 ± 0.08   
                            SVR                         0.20 ± 0.02   
                            Decision Tree              -0.27 ± 0.2